In [ ]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM, pipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# Step 1: Load the pretrained emotion recognition pipeline via Hugging Face
translation_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-uk-en")
translation_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-uk-en")

translation_pipe = pipeline('translation', model=translation_model, tokenizer=translation_tokenizer, device=device)

emotion_model = AutoModelForSequenceClassification.from_pretrained("j-hartmann/emotion-english-distilroberta-base")
emotion_tokenizer = AutoTokenizer.from_pretrained("j-hartmann/emotion-english-distilroberta-base")


emotion_pipe = pipeline('text-classification', model=emotion_model, tokenizer=emotion_tokenizer, device=device)


# Step 2: Define the function for emotion recognition
def predict_emotion(text: str) -> str:
   # Predict emotion probabilities
   english_text = translation_pipe(text, max_length=512)[0]['translation_text']

   predictions = emotion_pipe(english_text, top_k=7)
   # predictions is a list of dictionaries with 'label' and 'score'
   # e.g. [{'label': 'joy', 'score': 0.78}, ... ]

   # Find the emotion label with the highest score
   best_emotion = max(predictions, key=lambda x: x["score"])
   emotion_label = best_emotion["label"]
   confidence_score = best_emotion["score"]

   eng_emotion_to_ukr = {
       "joy": "радість",
       "anger": "гнів",
       "disgust": "відраза",
       "sadness": "сум",
       "fear": "страх",
       "surprise": "здивування",
       "neutral": "нейтральність"
   }

   # Return a formatted string with the top predicted emotion and confidence
   return f"{eng_emotion_to_ukr[emotion_label].upper()} (Впевненість: {confidence_score*100:.2f}%)"

# Step 3: Build the Gradio interface
interface = gr.Interface(
   fn=predict_emotion,
   inputs=gr.Textbox(lines=2, label="Введіть текст:"),
   outputs=gr.Text(lines=2, label="Результат розпізнавання:"),
   title="Розпізнавання Емоційного Забарвлення Тексту",
   description="Введіть текст для розпізнавання емоційного забарвлення",
   examples=['Я такий щасливий, накінецьто здійснилася моя мрія!', 'Моя мама в лікарні, я дуже хвилююсь за неї.']
)

# Step 4: Launch the interface
if __name__ == "__main__":
   interface.launch()
